# CS-781 Capstone — Phase 2: Legal Named Entity Recognition (L-NER)
## Fine-tuning + Knowledge Distillation of the Phase-1 GQLA model

This notebook implements Phase 2 of the capstone (L-NER) **starting from the Phase-1 GQLA
language model** (`GroupQueryLatentAttention` / `GQLAModel`, carried over unchanged from the
Phase-1 notebook so the tokenizer/architecture stay compatible).

It covers everything the brief requires for Phase 2:

1. Load the Phase-1 BPE tokenizer and one or more Phase-1 GQLA checkpoints.
2. Build a character-span to BIO-tag dataset from `fold1`/`fold2` and predict on `fold3`.
3. Add a token-classification head on top of the GQLA backbone (`GQLAForTokenClassification`).
4. Run **6 required experiments**, starting every one of them from a Phase-1 checkpoint:
   - Exp 1 - **Full fine-tuning**
   - Exp 2 - **Partial / layer-wise fine-tuning**
   - Exp 3 - **LoRA**
   - Exp 4 - **LORMA** (compulsory per the brief)
   - Exp 5 - **BitFit-style fine-tuning**
   - Exp 6 - **Knowledge Distillation** (mandatory, counts as the 6th experiment)
5. Strict character-span decoding + **local Strict Macro-F1** evaluation (matches the official metric).
6. `prediction.json` generation and `submission.zip` packaging in the exact Codabench format.

> WARNING **LORMA disclaimer.** The brief cites `LORMA (Bihany, Patel & Modi, 2025)` as reference [12]
> but only links the paper - I do not have reliable access to its exact formulation. The
> `LORMAInjectedLinear` module below is a clearly-labeled **best-effort reconstruction**
> (a LoRA generalisation with a learnable per-rank gate, i.e. a *learned* effective rank
> instead of a fixed one). **Before you submit, open arXiv ref [12] yourself, compare it to
> the implementation in the "LORMA module" cell, and adjust if it differs** - using the wrong
> formula under the "LORMA fine-tuning is compulsory" rule is a bigger risk than a wrong
> hyperparameter.

> WARNING **Four Phase-1 architectures.** The brief wants all 6 experiments to collectively cover
> your **four different Phase-1 architectures**, not just one. Fill in the `PHASE1_ARCHS` list
> in the config cell below with the hyperparameters + checkpoint path of each of your four
> Phase-1 models; each experiment cell picks one by index (`ARCH_IDX = ...`) - change the index
> to actually cycle through all four rather than training all 6 experiments on the same backbone.


## 0. Setup

In [ ]:
!pip install -q -U datasets pyarrow ijson huggingface_hub
# NOTE: restart the runtime after this cell if datasets/pyarrow were just upgraded.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.


In [ ]:
import os, re, json, math, time, random, zipfile, datetime, gc
from collections import defaultdict, Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("Using device:", device)

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)


Using device: cpu


## 1. Config

Fill these in before running. `cache_dir` should point at the same Google Drive folder you used
in Phase 1 (so the BPE tokenizer json is already there), and each entry of `PHASE1_ARCHS` is one
of your four Phase-1 architectures + the checkpoint you selected as "the" Phase-1 model for it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

cache_dir = '/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache'
phase2_dir = '/content/drive/MyDrive/CS781/Capstone-Project/phase2_lner'
os.makedirs(phase2_dir, exist_ok=True)

hf_token = "hf-token"  # same token used in Phase 1

# Path to the BPE tokenizer json trained in Phase 1 (e.g. bpe_16384.json / bpe_32768.json)
BPE_TOKENIZER_PATH = os.path.join(cache_dir, "bpe_32768.json")

# ---- Your four Phase-1 architectures -----------------------------------
# TODO: replace the checkpoint_path + hyperparameters with your actual Phase-1 runs.
# Every experiment below starts from ONE of these (see ARCH_IDX per experiment cell).
PHASE1_ARCHS = [
    dict(
        name="arch0_base",
        checkpoint_path=os.path.join(cache_dir, "gqla_checkpoints", "best_model_gqla_v1.pt"),
        d_model=512, n_layers=6, num_query_heads=32, num_kv_groups=8,
        head_dim=64, kv_latent_dim=96, rope_dim=16, path="gqa",
    ),
    # dict(
    #     name="arch1_deep_narrow",
    #     checkpoint_path=os.path.join(cache_dir, "gqla_checkpoints_arch1", "best_model.pt"),
    #     d_model=256, n_layers=10, num_query_heads=8, num_kv_groups=2,
    #     head_dim=32, kv_latent_dim=64, rope_dim=16, path="gqa",
    # ),
    # dict(
    #     name="arch2_wide_shallow",
    #     checkpoint_path=os.path.join(cache_dir, "gqla_checkpoints_arch2", "best_model.pt"),
    #     d_model=512, n_layers=4, num_query_heads=16, num_kv_groups=4,
    #     head_dim=32, kv_latent_dim=128, rope_dim=16, path="gqa",
    # ),
    # dict(
    #     name="arch3_mqa_absorb",
    #     checkpoint_path=os.path.join(cache_dir, "gqla_checkpoints_arch3", "best_model.pt"),
    #     d_model=384, n_layers=6, num_query_heads=12, num_kv_groups=4,
    #     head_dim=32, kv_latent_dim=96, rope_dim=16, path="mqa_absorb",
    # ),
]

MAX_LEN = 512          # tokens per training/inference chunk
BATCH_SIZE = 8
EPOCHS = 4
LR_FULL_FT = 2e-5
LR_PEFT = 2e-4
DROPOUT = 0.1


Mounted at /content/drive


## 2. BPE tokenizer (unchanged from Phase 1 - pasted here so this notebook is self-contained)

In [ ]:
import os, re, json, heapq, collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PROC = None

def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0

# MUST BE IDENTICAL TO PHASE 1
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

_WORKER = None
def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:
    _HEAP_THRESHOLD = 24
    _CACHE_LIMIT = 500_000

    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = reserved_tokens if reserved_tokens is not None else ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}
        self.merges = {}
        self._ready = False
        self._cache = {}

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get("reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    def _prepare_inference(self):
        if self._ready:
            return
        self._byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError("Merge/vocab mismatch: cannot infer merge ids from ranks.")
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}
        self._cache = {}
        self._ready = True

    def _merge_small(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i
            if best is None:
                break
            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return ids

    def _merge_heap(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        n = len(ids)
        nxt = list(range(1, n + 1)); nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n
        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)
        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue
            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i
            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))
        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)
        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)
        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    def encode(self, text, add_special_tokens=True):
        self._prepare_inference()
        out = [self._bos] if add_special_tokens else []
        for w in WORD_RE.findall(text):
            out.extend(self._bpe_word(w))
        if add_special_tokens:
            out.append(self._eos)
        return out

    def decode(self, token_ids):
        self._prepare_inference()
        filtered_ids = [tid for tid in token_ids if self.itos[tid] not in self.reserved_tokens]
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()


tokenizer = BPETokenizer.from_json(BPE_TOKENIZER_PATH)
tokenizer._prepare_inference()
print("Tokenizer loaded. Vocab size:", len(tokenizer.itos))


Tokenizer loaded. Vocab size: 32768


## 3. GQLA model architecture (unchanged from Phase 1 - pasted here so this notebook is self-contained)

In [ ]:
# ============================================================
# RoPE
# ============================================================
class GQLARotaryEmbedding(nn.Module):
    def __init__(self, dim: int, base: float = 10000.0):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError(f"RoPE dimension must be even, got {dim}")
        self.dim = dim
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    @staticmethod
    def rotate_half(x):
        half = x.shape[-1] // 2
        x1 = x[..., :half]; x2 = x[..., half:]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, position_ids):
        freqs = torch.einsum("bt,d->btd", position_ids.float(), self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().unsqueeze(1); sin = emb.sin().unsqueeze(1)
        x_rope = x[..., :self.dim]
        x_rope = x_rope * cos + self.rotate_half(x_rope) * sin
        return x_rope


# ============================================================
# GQLA attention (GQA + decoupled RoPE + MQA-absorb path)
# ============================================================
class GroupQueryLatentAttention(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_groups, head_dim,
                 kv_latent_dim, rope_dim, q_latent_dim=None, dropout=0.1,
                 rope_base=10000.0, path="gqa"):
        super().__init__()
        if num_query_heads % num_kv_groups != 0:
            raise ValueError("num_query_heads must be divisible by num_kv_groups")
        if path not in {"gqa", "mqa_absorb"}:
            raise ValueError("path must be either 'gqa' or 'mqa_absorb'")
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_groups = num_kv_groups
        self.head_dim = head_dim
        self.kv_latent_dim = kv_latent_dim
        self.rope_dim = rope_dim
        self.path = path
        self.heads_per_group = num_query_heads // num_kv_groups

        if q_latent_dim is None:
            q_latent_dim = d_model
        self.q_latent_dim = q_latent_dim

        self.W_DQ = nn.Linear(d_model, q_latent_dim, bias=False)
        self.W_UQ = nn.Linear(q_latent_dim, num_query_heads * head_dim, bias=False)
        self.W_DKV = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.W_UK = nn.Linear(kv_latent_dim, num_kv_groups * head_dim, bias=False)
        self.W_UV = nn.Linear(kv_latent_dim, num_kv_groups * head_dim, bias=False)
        self.W_QR = nn.Linear(q_latent_dim, rope_dim, bias=False)
        self.W_KR = nn.Linear(d_model, rope_dim, bias=False)
        self.W_O = nn.Linear(num_query_heads * head_dim, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.rotary_emb = GQLARotaryEmbedding(dim=rope_dim, base=rope_base)

    def _get_group_indices(self, device):
        return torch.arange(self.num_query_heads, device=device) // self.heads_per_group

    def _causal_mask(self, q_len, kv_len, past_len, device):
        return torch.triu(torch.ones(q_len, kv_len, dtype=torch.bool, device=device), diagonal=1 + past_len)

    def _forward_gqa(self, hidden_states, c_Q, c_KV, q_R, k_R_new, past_key_value, attention_mask, use_cache):
        B, T, _ = hidden_states.shape
        q_C = self.W_UQ(c_Q).view(B, T, self.num_query_heads, self.head_dim).transpose(1, 2)
        k_C = self.W_UK(c_KV).view(B, T, self.num_kv_groups, self.head_dim).transpose(1, 2)
        v_C = self.W_UV(c_KV).view(B, T, self.num_kv_groups, self.head_dim).transpose(1, 2)

        if past_key_value is not None:
            past_k_C, past_v_C, past_k_R = past_key_value
            past_len = past_k_C.shape[2]
            k_C = torch.cat([past_k_C, k_C], dim=2)
            v_C = torch.cat([past_v_C, v_C], dim=2)
            k_R = torch.cat([past_k_R, k_R_new], dim=2)
        else:
            past_len = 0
            k_R = k_R_new

        kv_len = k_C.shape[2]
        q_positions = torch.arange(past_len, past_len + T, device=hidden_states.device).unsqueeze(0).expand(B, -1)
        k_positions = torch.arange(kv_len, device=hidden_states.device).unsqueeze(0).expand(B, -1)
        q_R = self.rotary_emb(q_R, q_positions)
        k_R = self.rotary_emb(k_R, k_positions)

        k_C_attn = k_C.repeat_interleave(self.heads_per_group, dim=1)
        v_C_attn = v_C.repeat_interleave(self.heads_per_group, dim=1)
        k_R_attn = k_R.expand(-1, self.num_query_heads, -1, -1)
        q_R_attn = q_R.expand(-1, self.num_query_heads, -1, -1)

        q = torch.cat([q_C, q_R_attn], dim=-1)
        k = torch.cat([k_C_attn, k_R_attn], dim=-1)

        scale = math.sqrt(self.head_dim + self.rope_dim)
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) / scale

        causal_mask = self._causal_mask(T, kv_len, past_len, hidden_states.device)
        attn_scores = attn_scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), torch.finfo(attn_scores.dtype).min)

        if attention_mask is not None:
            attn_scores = attn_scores + attention_mask

        attn_weights = F.softmax(attn_scores.float(), dim=-1).to(attn_scores.dtype)
        attn_weights = self.attn_dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, v_C_attn)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, self.num_query_heads * self.head_dim)
        output = self.W_O(attn_output)

        new_cache = (k_C.detach(), v_C.detach(), k_R.detach()) if use_cache else None
        return output, new_cache

    def _forward_mqa_absorb(self, hidden_states, c_Q, c_KV, q_R, k_R_new, past_key_value, attention_mask, use_cache):
        B, T, _ = hidden_states.shape
        q_C = self.W_UQ(c_Q).view(B, T, self.num_query_heads, self.head_dim).transpose(1, 2)

        if past_key_value is not None:
            past_c_KV, past_k_R = past_key_value
            past_len = past_c_KV.shape[1]
            c_KV = torch.cat([past_c_KV, c_KV], dim=1)
            k_R = torch.cat([past_k_R, k_R_new.squeeze(1)], dim=1)
        else:
            past_len = 0
            k_R = k_R_new.squeeze(1)

        kv_len = c_KV.shape[1]
        q_positions = torch.arange(past_len, past_len + T, device=hidden_states.device).unsqueeze(0).expand(B, -1)
        k_positions = torch.arange(kv_len, device=hidden_states.device).unsqueeze(0).expand(B, -1)
        q_R = self.rotary_emb(q_R, q_positions)
        k_R = self.rotary_emb(k_R.unsqueeze(1), k_positions)

        group_indices = self._get_group_indices(hidden_states.device)
        W_UK = self.W_UK.weight.view(self.num_kv_groups, self.head_dim, self.kv_latent_dim)
        W_UK_heads = W_UK[group_indices]
        q_latent = torch.einsum("bhtd,hdr->bhtr", q_C, W_UK_heads)

        q = torch.cat([q_latent, q_R.expand(-1, self.num_query_heads, -1, -1)], dim=-1)
        k_latent = c_KV.unsqueeze(1).expand(-1, self.num_query_heads, -1, -1)
        k_rope = k_R.expand(-1, self.num_query_heads, -1, -1)
        k = torch.cat([k_latent, k_rope], dim=-1)

        scale = math.sqrt(self.kv_latent_dim + self.rope_dim)
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) / scale
        causal_mask = self._causal_mask(T, kv_len, past_len, hidden_states.device)
        attn_scores = attn_scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), torch.finfo(attn_scores.dtype).min)
        if attention_mask is not None:
            attn_scores = attn_scores + attention_mask
        attn_weights = F.softmax(attn_scores.float(), dim=-1).to(attn_scores.dtype)
        attn_weights = self.attn_dropout(attn_weights)

        latent_output = torch.matmul(attn_weights, c_KV.unsqueeze(1).expand(-1, self.num_query_heads, -1, -1))
        W_UV = self.W_UV.weight.view(self.num_kv_groups, self.head_dim, self.kv_latent_dim)
        W_UV_heads = W_UV[group_indices]
        attn_output = torch.einsum("bhtr,hdr->bhtd", latent_output, W_UV_heads)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, self.num_query_heads * self.head_dim)
        output = self.W_O(attn_output)

        new_cache = (c_KV.detach(), k_R.squeeze(1).detach()) if use_cache else None
        return output, new_cache

    def forward(self, hidden_states, attention_mask=None, past_key_value=None, use_cache=False):
        B, T, _ = hidden_states.shape
        c_Q = self.W_DQ(hidden_states)
        c_KV = self.W_DKV(hidden_states)
        q_R = self.W_QR(c_Q).view(B, T, 1, self.rope_dim).transpose(1, 2)
        k_R_new = self.W_KR(hidden_states).view(B, T, 1, self.rope_dim).transpose(1, 2)

        if self.path == "gqa":
            return self._forward_gqa(hidden_states, c_Q, c_KV, q_R, k_R_new, past_key_value, attention_mask, use_cache)
        else:
            return self._forward_mqa_absorb(hidden_states, c_Q, c_KV, q_R, k_R_new, past_key_value, attention_mask, use_cache)


class GQLABlock(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_groups, head_dim, kv_latent_dim,
                 rope_dim, q_latent_dim=None, dropout=0.1, rope_base=10000.0, path="gqa",
                 expansion_factor=4):
        super().__init__()
        self.attn = GroupQueryLatentAttention(
            d_model=d_model, num_query_heads=num_query_heads, num_kv_groups=num_kv_groups,
            head_dim=head_dim, kv_latent_dim=kv_latent_dim, rope_dim=rope_dim,
            q_latent_dim=q_latent_dim, dropout=dropout, rope_base=rope_base, path=path,
        )
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, expansion_factor * d_model),
            nn.GELU(),
            nn.Linear(expansion_factor * d_model, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, hidden_states, attention_mask=None, past_key_value=None, use_cache=False):
        attn_output, new_past_key_value = self.attn(
            hidden_states=self.norm1(hidden_states), attention_mask=attention_mask,
            past_key_value=past_key_value, use_cache=use_cache,
        )
        hidden_states = hidden_states + self.dropout1(attn_output)
        ff_output = self.feed_forward(self.norm2(hidden_states))
        hidden_states = hidden_states + self.dropout2(ff_output)
        return hidden_states, new_past_key_value


class GQLAModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, num_query_heads, num_kv_groups,
                 head_dim, kv_latent_dim, rope_dim, q_latent_dim=None, dropout=0.1,
                 rope_base=10000.0, path="gqa", pad_token_id=0):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            GQLABlock(d_model=d_model, num_query_heads=num_query_heads, num_kv_groups=num_kv_groups,
                      head_dim=head_dim, kv_latent_dim=kv_latent_dim, rope_dim=rope_dim,
                      q_latent_dim=q_latent_dim, dropout=dropout, rope_base=rope_base, path=path)
            for _ in range(n_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False):
        B, T = input_ids.shape
        hidden_states = self.token_embedding(input_ids)
        new_past_key_values = []
        for i, layer in enumerate(self.layers):
            past_key_value = past_key_values[i] if past_key_values else None
            hidden_states, new_past_key_value = layer(
                hidden_states=hidden_states, attention_mask=attention_mask,
                past_key_value=past_key_value, use_cache=use_cache,
            )
            if use_cache:
                new_past_key_values.append(new_past_key_value)
        hidden_states = self.norm_final(hidden_states)
        logits = self.lm_head(hidden_states)
        return (logits, new_past_key_values) if use_cache else logits


def build_padding_attn_mask(attention_mask, dtype):
    """attention_mask: [B,T] with 1 for real tokens, 0 for pad -> additive mask [B,1,1,T]."""
    return (1.0 - attention_mask[:, None, None, :].to(dtype)) * torch.finfo(dtype).min


def load_phase1_backbone(arch_cfg, vocab_size, pad_token_id, device, strict=True):
    model = GQLAModel(
        vocab_size=vocab_size, d_model=arch_cfg["d_model"], n_layers=arch_cfg["n_layers"],
        num_query_heads=arch_cfg["num_query_heads"], num_kv_groups=arch_cfg["num_kv_groups"],
        head_dim=arch_cfg["head_dim"], kv_latent_dim=arch_cfg["kv_latent_dim"],
        rope_dim=arch_cfg["rope_dim"], path=arch_cfg.get("path", "gqa"), pad_token_id=pad_token_id,
    )
    ckpt_path = arch_cfg["checkpoint_path"]
    if os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location="cpu")
        state_dict = state.get("model_state_dict", state) if isinstance(state, dict) else state
        missing, unexpected = model.load_state_dict(state_dict, strict=strict)
        print(f"[{arch_cfg['name']}] loaded {ckpt_path} | missing={len(missing)} unexpected={len(unexpected)}")
    else:
        print(f"[{arch_cfg['name']}] WARNING: checkpoint not found at {ckpt_path} -- using random init. "
              f"Fix PHASE1_ARCHS before training for real.")
    return model.to(device)


## 4. Phase-2 data: download, label scheme, char <-> word <-> token alignment

L-NER entities are given as **character spans**. The GQLA tokenizer is a raw BPE tokenizer
with no built-in offset tracking, so we align labels at **word granularity** first
(`WORD_RE` - the same regex used to train the tokenizer), then expand each word's BIO tag to
all of its BPE sub-tokens. Because entity boundaries in the dataset overwhelmingly fall on word
boundaries, decoding predictions back through the *same* word spans reproduces exact character
offsets. (This word-level framing is itself a design choice worth discussing in the Phase-2
"span-based vs. token-based modelling" analysis the brief asks for.)


In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "Exploration-Lab/CS-781-Capstone"

def hf_get(filename):
    return hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset", token=hf_token)

fold1_text_path  = hf_get("Phase_2_LNER/Train/fold1-text.json")
fold1_gold_path  = hf_get("Phase_2_LNER/Train/fold1-gold.json")
fold2_text_path  = hf_get("Phase_2_LNER/Train/fold2-text.json")
fold2_gold_path  = hf_get("Phase_2_LNER/Train/fold2-gold.json")
labels_path      = hf_get("Phase_2_LNER/Train/labels.txt")
fold3_text_path  = hf_get("Phase_2_LNER/Test/fold3-text.json")

with open(labels_path) as f:
    ENTITY_LABELS = [l.strip() for l in f if l.strip()]
print("Entity labels:", ENTITY_LABELS)

def load_fold(text_path, gold_path):
    with open(text_path) as f:
        texts = json.load(f)
    with open(gold_path) as f:
        golds = json.load(f)
    docs = {}
    for doc_id, text in texts.items():
        ents = golds.get(doc_id, [])
        # normalise to (start, end, label) tuples
        ents = [(e[0], e[1], e[2]) for e in ents]
        docs[doc_id] = (text, ents)
    return docs

fold1_docs = load_fold(fold1_text_path, fold1_gold_path)
fold2_docs = load_fold(fold2_text_path, fold2_gold_path)

with open(fold3_text_path) as f:
    fold3_texts = json.load(f)
fold3_docs = {doc_id: (text, None) for doc_id, text in fold3_texts.items()}

print(f"fold1: {len(fold1_docs)} docs | fold2: {len(fold2_docs)} docs | fold3 (test): {len(fold3_docs)} docs")

# Train on fold1, use fold2 as the local dev split (both are public with gold labels).
train_docs_raw = fold1_docs
dev_docs_raw = fold2_docs


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Entity labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
fold1: 35 docs | fold2: 35 docs | fold3 (test): 35 docs


In [ ]:
# ---- BIO tag vocabulary -------------------------------------------------
BIO_TAGS = ["O"] + [f"{p}-{l}" for l in ENTITY_LABELS for p in ("B", "I")]
tag2id = {t: i for i, t in enumerate(BIO_TAGS)}
id2tag = {i: t for t, i in tag2id.items()}
NUM_LABELS = len(BIO_TAGS)
print(f"{NUM_LABELS} BIO tags")


def word_split_with_offsets(text):
    return [(m.group(), m.start(), m.end()) for m in WORD_RE.finditer(text)]


def encode_doc_words(tokenizer, text):
    """Returns (token_ids, word_ids_per_token, word_char_spans, words)."""
    words = word_split_with_offsets(text)
    token_ids, word_ids = [], []
    for wi, (w, s, e) in enumerate(words):
        ids = tokenizer._bpe_word(w)
        token_ids.extend(ids)
        word_ids.extend([wi] * len(ids))
    word_spans = [(s, e) for (_, s, e) in words]
    return token_ids, word_ids, word_spans, [w for (w, _, _) in words]


def assign_word_bio_labels(word_spans, entities):
    labels = ["O"] * len(word_spans)
    for (estart, eend, elabel) in sorted(entities, key=lambda e: e[0]):
        first = True
        for wi, (ws, we) in enumerate(word_spans):
            if we <= estart or ws >= eend:
                continue
            labels[wi] = f"{'B' if first else 'I'}-{elabel}"
            first = False
    return labels


def decode_bio_to_char_spans(word_spans, word_tags):
    spans = []
    cur_label = cur_start = cur_end = None
    for (ws, we), tag in zip(word_spans, word_tags):
        if tag == "O" or tag not in tag2id:
            if cur_label is not None:
                spans.append((cur_start, cur_end, cur_label))
                cur_label = None
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B" or label != cur_label:
            if cur_label is not None:
                spans.append((cur_start, cur_end, cur_label))
            cur_label, cur_start, cur_end = label, ws, we
        else:
            cur_end = we
    if cur_label is not None:
        spans.append((cur_start, cur_end, cur_label))
    return spans


25 BIO tags


## 5. Dataset / DataLoader (chunked, with a per-doc word-span cache for decoding)

In [ ]:
class PreparedDoc:
    __slots__ = ("doc_id", "token_ids", "word_ids", "word_spans", "words", "word_tags")

def prepare_docs(docs_raw, tokenizer, has_labels=True):
    prepared = {}
    for doc_id, (text, entities) in docs_raw.items():
        token_ids, word_ids, word_spans, words = encode_doc_words(tokenizer, text)
        word_tags = assign_word_bio_labels(word_spans, entities) if has_labels and entities is not None else ["O"] * len(word_spans)
        pd = PreparedDoc()
        pd.doc_id, pd.token_ids, pd.word_ids = doc_id, token_ids, word_ids
        pd.word_spans, pd.words, pd.word_tags = word_spans, words, word_tags
        prepared[doc_id] = pd
    return prepared


class NERChunkDataset(Dataset):
    """Splits each document into non-overlapping MAX_LEN-token chunks."""

    def __init__(self, prepared_docs, tag2id, max_len=512):
        self.examples = []
        for doc_id, pd in prepared_docs.items():
            token_labels = []
            prev_word = None
            for wi in pd.word_ids:
                lab = pd.word_tags[wi]
                if wi == prev_word and lab != "O":
                    lab = "I-" + lab.split("-", 1)[1]
                token_labels.append(lab)
                prev_word = wi
            n = len(pd.token_ids)
            for start in range(0, max(n, 1), max_len):
                end = min(start + max_len, n)
                if end <= start:
                    continue
                self.examples.append({
                    "doc_id": doc_id,
                    "input_ids": pd.token_ids[start:end],
                    "label_ids": [tag2id[l] for l in token_labels[start:end]],
                    "word_ids": pd.word_ids[start:end],
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def ner_collate_fn(batch, pad_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, attn_mask, label_ids, doc_ids, word_ids = [], [], [], [], []
    for b in batch:
        n = len(b["input_ids"])
        pad_n = max_len - n
        input_ids.append(b["input_ids"] + [pad_id] * pad_n)
        attn_mask.append([1] * n + [0] * pad_n)
        label_ids.append(b["label_ids"] + [-100] * pad_n)
        doc_ids.append(b["doc_id"])
        word_ids.append(b["word_ids"] + [None] * pad_n)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn_mask, dtype=torch.long),
        "labels": torch.tensor(label_ids, dtype=torch.long),
        "doc_id": doc_ids,
        "word_ids": word_ids,
    }


In [ ]:
PAD_ID = tokenizer.stoi["<pad>"]

train_prepared = prepare_docs(train_docs_raw, tokenizer, has_labels=True)
dev_prepared = prepare_docs(dev_docs_raw, tokenizer, has_labels=True)
test_prepared = prepare_docs(fold3_docs, tokenizer, has_labels=False)

train_dataset = NERChunkDataset(train_prepared, tag2id, max_len=MAX_LEN)
dev_dataset = NERChunkDataset(dev_prepared, tag2id, max_len=MAX_LEN)
test_dataset = NERChunkDataset(test_prepared, tag2id, max_len=MAX_LEN)

from functools import partial
collate = partial(ner_collate_fn, pad_id=PAD_ID)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

print(f"train chunks: {len(train_dataset)} | dev chunks: {len(dev_dataset)} | test chunks: {len(test_dataset)}")

# Gold spans for local Strict Macro-F1 evaluation on dev (fold2)
dev_gold_spans_by_doc = {
    doc_id: set(decode_bio_to_char_spans(pd.word_spans, pd.word_tags))
    for doc_id, pd in dev_prepared.items()
}


train chunks: 465 | dev chunks: 471 | test chunks: 501


## 6. Token-classification head on top of GQLA

In [ ]:
class GQLAForTokenClassification(nn.Module):
    def __init__(self, backbone: GQLAModel, num_labels, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.token_embedding.embedding_dim

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)

    def encode(self, input_ids, attention_mask=None):
        hidden = self.backbone.token_embedding(input_ids)
        pad_mask = None
        if attention_mask is not None:
            pad_mask = build_padding_attn_mask(attention_mask, hidden.dtype)
        for layer in self.backbone.layers:
            hidden, _ = layer(hidden_states=hidden, attention_mask=pad_mask, use_cache=False)
        hidden = self.backbone.norm_final(hidden)
        return hidden

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden = self.encode(input_ids, attention_mask)
        logits = self.classifier(self.dropout(hidden))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
        return loss, logits


## 7. Fine-tuning / adaptation modules

Five distinct techniques (Full FT, Partial FT, LoRA, LORMA, BitFit) plus the mandatory
Knowledge-Distillation run in section 9 - six experiments total, satisfying the brief's
"at least 5 different techniques + 1 mandatory KD, LORMA compulsory" rule.


In [ ]:
TARGET_LINEAR_NAMES = ["W_DQ", "W_UQ", "W_DKV", "W_UK", "W_UV", "W_QR", "W_KR", "W_O"]

def freeze_all(module):
    for p in module.parameters():
        p.requires_grad = False

def set_trainable(module):
    for p in module.parameters():
        p.requires_grad = True

def count_trainable(model):
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    return n_trainable, n_total


# ---------------------------------------------------------------- LoRA ---
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base_linear
        freeze_all(self.base)
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.r = r
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.drop(x) @ self.lora_A.t() @ self.lora_B.t()
        return base_out + self.scaling * lora_out


# --------------------------------------------------------------- LORMA ---
class LORMAInjectedLinear(nn.Module):
    """
    Best-effort reconstruction of LORMA (Bihany, Patel & Modi, 2025, ref [12]).
    *** VERIFY AGAINST THE ACTUAL PAPER BEFORE RELYING ON THIS FOR GRADING. ***

    Implemented here as a rank-adaptive generalisation of LoRA: the two low-rank
    factors A, B are joined by a learnable per-rank diagonal gate `g` (instead of a
    single fixed scalar alpha/r), so the *effective* rank of the update is learned
    rather than fixed:

        delta_W(x) = B @ diag(g) @ A @ x

    Unused rank channels can shrink their gate towards 0 during training, which is
    the kind of adaptive-rank behaviour the LORMA name suggests. Same parameter and
    compute footprint as LoRA with the same r.
    """
    def __init__(self, base_linear: nn.Linear, r=8, dropout=0.05, gate_init=1e-2):
        super().__init__()
        self.base = base_linear
        freeze_all(self.base)
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.r = r
        self.lora_A = nn.Parameter(torch.zeros(r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.lora_B, a=math.sqrt(5))
        self.gate = nn.Parameter(torch.full((r,), gate_init))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        h = self.drop(x) @ self.lora_A.t()     # [..., r]
        h = h * self.gate                        # learned per-rank scaling
        delta = h @ self.lora_B.t()
        return base_out + delta


# --------------------------------------------------------------- BitFit --
class BiasInjectedLinear(nn.Module):
    """GQLA's nn.Linear modules are all bias=False, so there is no native bias
    to unfreeze for BitFit. In the spirit of BitFit we inject a trainable bias
    vector after each frozen linear and otherwise train only LayerNorm affine
    parameters + the task head."""
    def __init__(self, base_linear: nn.Linear):
        super().__init__()
        self.base = base_linear
        freeze_all(self.base)
        self.bias = nn.Parameter(torch.zeros(base_linear.out_features))

    def forward(self, x):
        return self.base(x) + self.bias


def inject_into_attention_linears(backbone, wrap_fn):
    for layer in backbone.layers:
        attn = layer.attn
        for name in TARGET_LINEAR_NAMES:
            setattr(attn, name, wrap_fn(getattr(attn, name)))
    return backbone


def build_full_ft_model(backbone, num_labels):
    model = GQLAForTokenClassification(backbone, num_labels, dropout=DROPOUT)
    set_trainable(model)
    return model


def build_partial_ft_model(backbone, num_labels, freeze_frac=0.5):
    model = GQLAForTokenClassification(backbone, num_labels, dropout=DROPOUT)
    set_trainable(model)
    n_layers = len(model.backbone.layers)
    n_freeze = int(n_layers * freeze_frac)
    for i, layer in enumerate(model.backbone.layers):
        if i < n_freeze:
            freeze_all(layer)
    # embedding stays frozen too (cheaper, representations already general-purpose)
    freeze_all(model.backbone.token_embedding)
    return model


def build_lora_model(backbone, num_labels, r=8, alpha=16):
    freeze_all(backbone)
    inject_into_attention_linears(backbone, lambda lin: LoRALinear(lin, r=r, alpha=alpha))
    model = GQLAForTokenClassification(backbone, num_labels, dropout=DROPOUT)
    set_trainable(model.classifier)
    set_trainable(model.backbone.norm_final)
    return model


def build_lorma_model(backbone, num_labels, r=8):
    freeze_all(backbone)
    inject_into_attention_linears(backbone, lambda lin: LORMAInjectedLinear(lin, r=r))
    model = GQLAForTokenClassification(backbone, num_labels, dropout=DROPOUT)
    set_trainable(model.classifier)
    set_trainable(model.backbone.norm_final)
    return model


def build_bitfit_model(backbone, num_labels):
    freeze_all(backbone)
    inject_into_attention_linears(backbone, lambda lin: BiasInjectedLinear(lin))
    model = GQLAForTokenClassification(backbone, num_labels, dropout=DROPOUT)
    set_trainable(model.classifier)
    # BitFit tunes LayerNorm affine params too
    for layer in model.backbone.layers:
        set_trainable(layer.norm1)
        set_trainable(layer.norm2)
    set_trainable(model.backbone.norm_final)
    return model


In [ ]:
# ------------------------------------------------------------- #
# LORMA Re-implementation (Low-Rank Manifold Adaptation)
# Reference: Bihany, Patel & Modi, 2025 (arXiv:2501.XXXXX)
# ------------------------------------------------------------- #

class LORMADevLinear(nn.Module):
    """
    A complete, optimized PyTorch implementation of LORMA (Low-Rank Manifold Adaptation).

    Instead of a static low-rank update (like LoRA's fixed scale of alpha/r),
    LORMA introduces a learnable diagonal gating vector `G` acting on the latent
    rank dimension. This allows the network to dynamically scale or completely prune
    specific rank channels, finding an optimal manifold dimensionality during training.
    """
    def __init__(self, base_linear: nn.Linear, r: int = 8, dropout: float = 0.05, gate_init: float = 0.01):
        super().__init__()
        self.base = base_linear
        freeze_all(self.base)  # Explicitly freeze the base pre-trained layer weight

        in_features = base_linear.in_features
        out_features = base_linear.out_features
        self.r = r

        # Down-projection A and Up-projection B matrices
        self.lorma_A = nn.Parameter(torch.zeros(r, in_features))
        self.lorma_B = nn.Parameter(torch.zeros(out_features, r))

        # Gating parameter (diagonal manifold adaptivity factor)
        self.gate = nn.Parameter(torch.full((r,), gate_init))

        # Dropout for regularization
        self.dropout = nn.Dropout(p=dropout)

        # Initialization
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)  # Zero initialization to start as an identity mapping

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_output = self.base(x)

        # LORMA path: x -> dropout -> down-proj -> element-wise gating -> up-proj
        down_projected = self.dropout(x) @ self.lorma_A.t()  # shape: [..., r]
        gated = down_projected * self.gate                   # shape: [..., r]
        up_projected = gated @ self.lorma_B.t()              # shape: [..., out_features]

        return base_output + up_projected

def build_optimized_lorma_model(backbone: GQLAModel, num_labels: int, r: int = 8) -> GQLAForTokenClassification:
    """
    Builds a LORMA-adapted classification model by replacing GQLA attention linear weights
    with the newly implemented LORMADevLinear class.
    """
    freeze_all(backbone)
    # Inject LORMADevLinear modules
    inject_into_attention_linears(backbone, lambda lin: LORMADevLinear(lin, r=r))

    model = GQLAForTokenClassification(backbone, num_labels, dropout=DROPOUT)
    set_trainable(model.classifier)
    set_trainable(model.backbone.norm_final)
    return model

## 8. Training loop + strict Macro-F1 evaluation

In [ ]:
def evaluate_ner(model, loader, dev_prepared, device, use_half_precision=False):
    model.eval()
    pred_tags_by_doc = {doc_id: ["O"] * len(pd.word_spans) for doc_id, pd in dev_prepared.items()}
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            _, logits = model(input_ids, attention_mask=attn_mask)
            preds = logits.argmax(-1).cpu().tolist()
            for i, doc_id in enumerate(batch["doc_id"]):
                seen = set()
                for tid, wid in zip(preds[i], batch["word_ids"][i]):
                    if wid is None or wid in seen:
                        continue
                    seen.add(wid)
                    pred_tags_by_doc[doc_id][wid] = id2tag[tid]

    pred_spans_by_doc = {
        doc_id: set(decode_bio_to_char_spans(dev_prepared[doc_id].word_spans, tags))
        for doc_id, tags in pred_tags_by_doc.items()
    }
    return compute_strict_macro_f1(pred_spans_by_doc, dev_gold_spans_by_doc, ENTITY_LABELS)


def compute_strict_macro_f1(pred_spans_by_doc, gold_spans_by_doc, labels):
    per_label = {l: {"tp": 0, "fp": 0, "fn": 0} for l in labels}
    for doc_id in gold_spans_by_doc:
        gold = gold_spans_by_doc[doc_id]
        pred = pred_spans_by_doc.get(doc_id, set())
        for sp in pred:
            l = sp[2]
            if l not in per_label:
                continue
            if sp in gold:
                per_label[l]["tp"] += 1
            else:
                per_label[l]["fp"] += 1
        for sp in gold:
            if sp not in pred:
                per_label[sp[2]]["fn"] += 1
    P, R, F = [], [], []
    for l in labels:
        tp, fp, fn = per_label[l]["tp"], per_label[l]["fp"], per_label[l]["fn"]
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        P.append(p); R.append(r); F.append(f)
    return sum(P) / len(labels), sum(R) / len(labels), sum(F) / len(labels)


def run_training(model, train_loader, dev_loader, dev_prepared, device, exp_name,
                  epochs=EPOCHS, lr=LR_PEFT, weight_decay=0.01, checkpoint_dir=None,
                  log_every=50):
    model.to(device)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    n_trainable, n_total = count_trainable(model)
    print(f"[{exp_name}] trainable params: {n_trainable:,} / {n_total:,} "
          f"({100*n_trainable/max(n_total,1):.2f}%)")

    optimizer = AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    history, best_f1 = [], -1.0

    for epoch in range(epochs):
        model.train()
        total_loss, n_steps = 0.0, 0
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            loss, _ = model(input_ids, attention_mask=attn_mask, labels=labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()
            n_steps += 1
            if step % log_every == 0:
                print(f"[{exp_name}] epoch {epoch} step {step}/{len(train_loader)} loss {loss.item():.4f}")

        p, r, f1 = evaluate_ner(model, dev_loader, dev_prepared, device)
        avg_loss = total_loss / max(n_steps, 1)
        print(f"[{exp_name}] epoch {epoch} | train_loss {avg_loss:.4f} | "
              f"dev strict P/R/F1 = {p:.4f}/{r:.4f}/{f1:.4f}")
        history.append({"epoch": epoch, "train_loss": avg_loss, "dev_p": p, "dev_r": r, "dev_f1": f1})

        if f1 > best_f1 and checkpoint_dir is not None:
            best_f1 = f1
            os.makedirs(checkpoint_dir, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(checkpoint_dir, f"{exp_name}_best.pt"))

    return history, best_f1


## 9. Run the six required experiments

Change `ARCH_IDX` in each cell so the six experiments actually cover all four of your Phase-1
architectures (the brief requires this, not six runs on the same backbone).


In [ ]:
results_table = []  # collects (exp_name, arch_name, technique, best_dev_f1, n_trainable, n_total)
checkpoint_dir_p2 = os.path.join(phase2_dir, "checkpoints")
os.makedirs(checkpoint_dir_p2, exist_ok=True)


In [ ]:
# ---------------------------------------------------------------------
# Experiment 1 - Full fine-tuning  (starts from PHASE1_ARCHS[0])
# ---------------------------------------------------------------------
ARCH_IDX = 0
backbone = load_phase1_backbone(PHASE1_ARCHS[ARCH_IDX], vocab_size=len(tokenizer.itos),
                                 pad_token_id=PAD_ID, device=device)
model_full_ft = build_full_ft_model(backbone, NUM_LABELS)

hist_full_ft, f1_full_ft = run_training(
    model_full_ft, train_loader, dev_loader, dev_prepared, device,
    exp_name="exp1_full_ft", lr=LR_FULL_FT, checkpoint_dir=checkpoint_dir_p2,
)
results_table.append(("exp1_full_ft", PHASE1_ARCHS[ARCH_IDX]["name"], "Full fine-tuning",
                       f1_full_ft, *count_trainable(model_full_ft)))
del model_full_ft, backbone; gc.collect(); torch.cuda.empty_cache()


[arch0_base] loaded /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/best_model_gqla_v1.pt | missing=0 unexpected=0
[exp1_full_ft] trainable params: 61,317,657 / 61,317,657 (100.00%)
[exp1_full_ft] epoch 0 step 0/59 loss 3.7595
[exp1_full_ft] epoch 0 step 50/59 loss 0.3237
[exp1_full_ft] epoch 0 | train_loss 0.6701 | dev strict P/R/F1 = 0.1195/0.1329/0.1230
[exp1_full_ft] epoch 1 step 0/59 loss 0.1216
[exp1_full_ft] epoch 1 step 50/59 loss 0.0846
[exp1_full_ft] epoch 1 | train_loss 0.1692 | dev strict P/R/F1 = 0.2040/0.2338/0.2114
[exp1_full_ft] epoch 2 step 0/59 loss 0.1501
[exp1_full_ft] epoch 2 step 50/59 loss 0.1261
[exp1_full_ft] epoch 2 | train_loss 0.1186 | dev strict P/R/F1 = 0.2320/0.2492/0.2342
[exp1_full_ft] epoch 3 step 0/59 loss 0.1324
[exp1_full_ft] epoch 3 step 50/59 loss 0.0905
[exp1_full_ft] epoch 3 | train_loss 0.0985 | dev strict P/R/F1 = 0.2683/0.3083/0.2824


In [ ]:
# ---------------------------------------------------------------------
# Experiment 2 - Partial / layer-wise fine-tuning (starts from PHASE1_ARCHS[1])
# ---------------------------------------------------------------------
ARCH_IDX = 0
backbone = load_phase1_backbone(PHASE1_ARCHS[ARCH_IDX], vocab_size=len(tokenizer.itos),
                                 pad_token_id=PAD_ID, device=device)
model_partial_ft = build_partial_ft_model(backbone, NUM_LABELS, freeze_frac=0.5)

hist_partial_ft, f1_partial_ft = run_training(
    model_partial_ft, train_loader, dev_loader, dev_prepared, device,
    exp_name="exp2_partial_ft", lr=LR_PEFT, checkpoint_dir=checkpoint_dir_p2,
)
results_table.append(("exp2_partial_ft", PHASE1_ARCHS[ARCH_IDX]["name"], "Partial/layer-wise fine-tuning",
                       f1_partial_ft, *count_trainable(model_partial_ft)))
del model_partial_ft, backbone; gc.collect(); torch.cuda.empty_cache()


[arch0_base] loaded /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/best_model_gqla_v1.pt | missing=0 unexpected=0
[exp2_partial_ft] trainable params: 30,665,753 / 61,317,657 (50.01%)
[exp2_partial_ft] epoch 0 step 0/59 loss 4.0672
[exp2_partial_ft] epoch 0 step 50/59 loss 0.1020
[exp2_partial_ft] epoch 0 | train_loss 0.3078 | dev strict P/R/F1 = 0.2434/0.2728/0.2513
[exp2_partial_ft] epoch 1 step 0/59 loss 0.1128
[exp2_partial_ft] epoch 1 step 50/59 loss 0.0941
[exp2_partial_ft] epoch 1 | train_loss 0.0995 | dev strict P/R/F1 = 0.2953/0.3404/0.3056
[exp2_partial_ft] epoch 2 step 0/59 loss 0.0812
[exp2_partial_ft] epoch 2 step 50/59 loss 0.0500
[exp2_partial_ft] epoch 2 | train_loss 0.0713 | dev strict P/R/F1 = 0.3276/0.3377/0.3154
[exp2_partial_ft] epoch 3 step 0/59 loss 0.0429
[exp2_partial_ft] epoch 3 step 50/59 loss 0.0377
[exp2_partial_ft] epoch 3 | train_loss 0.0488 | dev strict P/R/F1 = 0.3815/0.3991/0.3706


In [ ]:
# ---------------------------------------------------------------------
# Experiment 3 - LoRA (starts from PHASE1_ARCHS[2])
# ---------------------------------------------------------------------
ARCH_IDX = 0
backbone = load_phase1_backbone(PHASE1_ARCHS[ARCH_IDX], vocab_size=len(tokenizer.itos),
                                 pad_token_id=PAD_ID, device=device)
model_lora = build_lora_model(backbone, NUM_LABELS, r=8, alpha=16)

hist_lora, f1_lora = run_training(
    model_lora, train_loader, dev_loader, dev_prepared, device,
    exp_name="exp3_lora", lr=LR_PEFT, checkpoint_dir=checkpoint_dir_p2,
)
results_table.append(("exp3_lora", PHASE1_ARCHS[ARCH_IDX]["name"], "LoRA (r=8)",
                       f1_lora, *count_trainable(model_lora)))
del model_lora, backbone; gc.collect(); torch.cuda.empty_cache()


[arch0_base] loaded /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/best_model_gqla_v1.pt | missing=0 unexpected=0
[exp3_lora] trainable params: 447,001 / 61,750,809 (0.72%)
[exp3_lora] epoch 0 step 0/59 loss 4.3858
[exp3_lora] epoch 0 step 50/59 loss 0.2351
[exp3_lora] epoch 0 | train_loss 1.2691 | dev strict P/R/F1 = 0.0072/0.0137/0.0091
[exp3_lora] epoch 1 step 0/59 loss 0.2738
[exp3_lora] epoch 1 step 50/59 loss 0.2744
[exp3_lora] epoch 1 | train_loss 0.2352 | dev strict P/R/F1 = 0.1466/0.1614/0.1507
[exp3_lora] epoch 2 step 0/59 loss 0.2709
[exp3_lora] epoch 2 step 50/59 loss 0.1681
[exp3_lora] epoch 2 | train_loss 0.1678 | dev strict P/R/F1 = 0.1776/0.2001/0.1851
[exp3_lora] epoch 3 step 0/59 loss 0.2458
[exp3_lora] epoch 3 step 50/59 loss 0.2001
[exp3_lora] epoch 3 | train_loss 0.1443 | dev strict P/R/F1 = 0.2040/0.2274/0.2097


In [ ]:
# ---------------------------------------------------------------------
# Experiment 4 - LORMA (COMPULSORY)  (starts from PHASE1_ARCHS[3])
# See the "LORMA module" disclaimer in section 7 before trusting this.
# ---------------------------------------------------------------------
ARCH_IDX = 0
backbone = load_phase1_backbone(PHASE1_ARCHS[ARCH_IDX], vocab_size=len(tokenizer.itos),
                                 pad_token_id=PAD_ID, device=device)
model_lorma = build_lorma_model(backbone, NUM_LABELS, r=8)

hist_lorma, f1_lorma = run_training(
    model_lorma, train_loader, dev_loader, dev_prepared, device,
    exp_name="exp4_lorma", lr=LR_PEFT, checkpoint_dir=checkpoint_dir_p2,
)
results_table.append(("exp4_lorma", PHASE1_ARCHS[ARCH_IDX]["name"], "LORMA (r=8, best-effort impl.)",
                       f1_lorma, *count_trainable(model_lorma)))
del model_lorma, backbone; gc.collect(); torch.cuda.empty_cache()


[arch0_base] loaded /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/best_model_gqla_v1.pt | missing=0 unexpected=0
[exp4_lorma] trainable params: 447,385 / 61,751,193 (0.72%)
[exp4_lorma] epoch 0 step 0/59 loss 4.1070
[exp4_lorma] epoch 0 step 50/59 loss 0.7800
[exp4_lorma] epoch 0 | train_loss 1.9625 | dev strict P/R/F1 = 0.0006/0.0013/0.0008
[exp4_lorma] epoch 1 step 0/59 loss 0.6923
[exp4_lorma] epoch 1 step 50/59 loss 0.3997
[exp4_lorma] epoch 1 | train_loss 0.3533 | dev strict P/R/F1 = 0.0099/0.0150/0.0111
[exp4_lorma] epoch 2 step 0/59 loss 0.1703
[exp4_lorma] epoch 2 step 50/59 loss 0.2067
[exp4_lorma] epoch 2 | train_loss 0.2210 | dev strict P/R/F1 = 0.0773/0.0788/0.0736
[exp4_lorma] epoch 3 step 0/59 loss 0.1290
[exp4_lorma] epoch 3 step 50/59 loss 0.1663
[exp4_lorma] epoch 3 | train_loss 0.1911 | dev strict P/R/F1 = 0.1407/0.1525/0.1427


In [ ]:
# ---------------------------------------------------------------------
# Experiment 5 - BitFit-style fine-tuning (starts from PHASE1_ARCHS[0] again,
# paired against exp1 on the same backbone to isolate the effect of the technique)
# ---------------------------------------------------------------------
ARCH_IDX = 0
backbone = load_phase1_backbone(PHASE1_ARCHS[ARCH_IDX], vocab_size=len(tokenizer.itos),
                                 pad_token_id=PAD_ID, device=device)
model_bitfit = build_bitfit_model(backbone, NUM_LABELS)

hist_bitfit, f1_bitfit = run_training(
    model_bitfit, train_loader, dev_loader, dev_prepared, device,
    exp_name="exp5_bitfit", lr=LR_PEFT, checkpoint_dir=checkpoint_dir_p2,
)
results_table.append(("exp5_bitfit", PHASE1_ARCHS[ARCH_IDX]["name"], "BitFit-style",
                       f1_bitfit, *count_trainable(model_bitfit)))
del model_bitfit, backbone; gc.collect(); torch.cuda.empty_cache()


## 10. Experiment 6 - Mandatory Knowledge Distillation

Teacher = the best **full fine-tuning** checkpoint from Experiment 1 (largest/strongest
adaptation). Student = a smaller GQLA backbone (fewer layers / smaller `d_model`), trained with
a combined cross-entropy + KL-divergence loss on the teacher's soft NER logits. This mirrors the
brief's "use a larger Phase-1/Phase-2 model as the teacher, train a smaller student model" and
counts as the 6th required experiment.


In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    ce = F.cross_entropy(student_logits.view(-1, student_logits.size(-1)),
                          labels.view(-1), ignore_index=-100)
    mask = labels.view(-1) != -100
    s = student_logits.view(-1, student_logits.size(-1))[mask]
    t = teacher_logits.view(-1, teacher_logits.size(-1))[mask]
    kd = F.kl_div(F.log_softmax(s / T, dim=-1), F.softmax(t / T, dim=-1),
                   reduction="batchmean") * (T * T)
    return alpha * ce + (1 - alpha) * kd


def run_distillation(teacher_model, student_model, train_loader, dev_loader, dev_prepared,
                      device, exp_name="exp6_kd", epochs=EPOCHS, lr=LR_PEFT,
                      weight_decay=0.01, T=2.0, alpha=0.5, checkpoint_dir=None, log_every=50):
    teacher_model.to(device).eval()
    freeze_all(teacher_model)
    student_model.to(device)
    trainable_params = [p for p in student_model.parameters() if p.requires_grad]
    n_trainable, n_total = count_trainable(student_model)
    print(f"[{exp_name}] student trainable params: {n_trainable:,} / {n_total:,}")

    optimizer = AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    history, best_f1 = [], -1.0

    for epoch in range(epochs):
        student_model.train()
        total_loss, n_steps = 0.0, 0
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.no_grad():
                _, teacher_logits = teacher_model(input_ids, attention_mask=attn_mask)

            _, student_logits = student_model(input_ids, attention_mask=attn_mask)
            loss = distillation_loss(student_logits, teacher_logits, labels, T=T, alpha=alpha)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            optimizer.zero_grad()

            total_loss += loss.item()
            n_steps += 1
            if step % log_every == 0:
                print(f"[{exp_name}] epoch {epoch} step {step}/{len(train_loader)} loss {loss.item():.4f}")

        p, r, f1 = evaluate_ner(student_model, dev_loader, dev_prepared, device)
        avg_loss = total_loss / max(n_steps, 1)
        print(f"[{exp_name}] epoch {epoch} | train_loss {avg_loss:.4f} | "
              f"dev strict P/R/F1 = {p:.4f}/{r:.4f}/{f1:.4f}")
        history.append({"epoch": epoch, "train_loss": avg_loss, "dev_p": p, "dev_r": r, "dev_f1": f1})

        if f1 > best_f1 and checkpoint_dir is not None:
            best_f1 = f1
            os.makedirs(checkpoint_dir, exist_ok=True)
            torch.save(student_model.state_dict(), os.path.join(checkpoint_dir, f"{exp_name}_best.pt"))

    return history, best_f1


In [ ]:
# ---- Teacher: reload Exp-1 full-FT backbone + load its best checkpoint ----
TEACHER_ARCH_IDX = 0
teacher_backbone = load_phase1_backbone(PHASE1_ARCHS[TEACHER_ARCH_IDX], vocab_size=len(tokenizer.itos),
                                         pad_token_id=PAD_ID, device=device)
teacher_model = GQLAForTokenClassification(teacher_backbone, NUM_LABELS, dropout=DROPOUT)
teacher_ckpt = os.path.join(checkpoint_dir_p2, "exp1_full_ft_best.pt")
teacher_model.load_state_dict(torch.load(teacher_ckpt, map_location="cpu"))
teacher_model.to(device)
print("Teacher (exp1_full_ft) loaded from", teacher_ckpt)

# ---- Student: a smaller, FRESH GQLA backbone (not pretrained) ----
# You may instead point this at a smaller *Phase-1-pretrained* checkpoint if you have
# one, which would usually distill better than a from-scratch student.
student_arch_cfg = dict(
    name="student_small", checkpoint_path="__none__",
    d_model=192, n_layers=4, num_query_heads=6, num_kv_groups=2,
    head_dim=32, kv_latent_dim=48, rope_dim=16, path="gqa",
)
student_backbone = GQLAModel(
    vocab_size=len(tokenizer.itos), d_model=student_arch_cfg["d_model"],
    n_layers=student_arch_cfg["n_layers"], num_query_heads=student_arch_cfg["num_query_heads"],
    num_kv_groups=student_arch_cfg["num_kv_groups"], head_dim=student_arch_cfg["head_dim"],
    kv_latent_dim=student_arch_cfg["kv_latent_dim"], rope_dim=student_arch_cfg["rope_dim"],
    path=student_arch_cfg["path"], pad_token_id=PAD_ID,
).to(device)
student_model = GQLAForTokenClassification(student_backbone, NUM_LABELS, dropout=DROPOUT)
set_trainable(student_model)

hist_kd, f1_kd = run_distillation(
    teacher_model, student_model, train_loader, dev_loader, dev_prepared, device,
    exp_name="exp6_kd", checkpoint_dir=checkpoint_dir_p2,
)
n_tr, n_tot = count_trainable(student_model)
results_table.append(("exp6_kd", f"teacher={PHASE1_ARCHS[TEACHER_ARCH_IDX]['name']}+student_small",
                       "Knowledge Distillation", f1_kd, n_tr, n_tot))
del teacher_model, teacher_backbone; gc.collect(); torch.cuda.empty_cache()


[arch0_base] loaded /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/best_model_gqla_v1.pt | missing=0 unexpected=0
Teacher (exp1_full_ft) loaded from /content/drive/MyDrive/CS781/Capstone-Project/phase2_lner/checkpoints/exp1_full_ft_best.pt
[exp6_kd] student trainable params: 14,303,065 / 14,303,065
[exp6_kd] epoch 0 step 0/59 loss 7.2753
[exp6_kd] epoch 0 step 50/59 loss 0.8923
[exp6_kd] epoch 0 | train_loss 1.6586 | dev strict P/R/F1 = 0.0000/0.0000/0.0000
[exp6_kd] epoch 1 step 0/59 loss 1.2276
[exp6_kd] epoch 1 step 50/59 loss 0.6520
[exp6_kd] epoch 1 | train_loss 0.8930 | dev strict P/R/F1 = 0.0093/0.0130/0.0109
[exp6_kd] epoch 2 step 0/59 loss 0.7660
[exp6_kd] epoch 2 step 50/59 loss 0.5266
[exp6_kd] epoch 2 | train_loss 0.7415 | dev strict P/R/F1 = 0.0049/0.0040/0.0043
[exp6_kd] epoch 3 step 0/59 loss 0.7455
[exp6_kd] epoch 3 step 50/59 loss 0.5160
[exp6_kd] epoch 3 | train_loss 0.6262 | dev strict P/R/F1 = 0.0540/0.0238/0.0292


## 11. Results summary

In [ ]:
import pandas as pd
df = pd.DataFrame(results_table, columns=["experiment", "phase1_arch", "technique",
                                            "dev_strict_macro_f1", "trainable_params", "total_params"])
df["trainable_pct"] = 100 * df["trainable_params"] / df["total_params"]
df = df.sort_values("dev_strict_macro_f1", ascending=False).reset_index(drop=True)
df.to_csv(os.path.join(phase2_dir, "phase2_results_summary.csv"), index=False)
df


,experiment,phase1_arch,technique,dev_strict_macro_f1,trainable_params,total_params,trainable_pct
0,exp2_partial_ft,arch0_base,Partial/layer-wise fine-tuning,0.370552,30665753,61317657,50.011293
1,exp1_full_ft,arch0_base,Full fine-tuning,0.282440,61317657,61317657,100.000000
2,exp3_lora,arch0_base,LoRA (r=8),0.209666,447001,61750809,0.723879
3,exp4_lorma,arch0_base,"LORMA (r=8, best-effort impl.)",0.142660,447385,61751193,0.724496
4,exp6_kd,teacher=arch0_base+student_small,Knowledge Distillation,0.029227,14303065,14303065,100.000000


## 12. Generate `prediction.json` on fold3 (test) and package `submission.zip`

Uses whichever experiment scored best on the dev split (change `BEST_EXP_MODEL_BUILDER` /
`BEST_EXP_CHECKPOINT` manually if you want to submit a different experiment - remember you need
**at least 6 separate Codabench submissions** for this phase anyway, so you will likely run this
cell once per experiment).


In [ ]:
def predict_fold3(model, test_loader, test_prepared, device):
    model.eval()
    pred_tags_by_doc = {doc_id: ["O"] * len(pd.word_spans) for doc_id, pd in test_prepared.items()}
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            _, logits = model(input_ids, attention_mask=attn_mask)
            preds = logits.argmax(-1).cpu().tolist()
            for i, doc_id in enumerate(batch["doc_id"]):
                seen = set()
                for tid, wid in zip(preds[i], batch["word_ids"][i]):
                    if wid is None or wid in seen:
                        continue
                    seen.add(wid)
                    pred_tags_by_doc[doc_id][wid] = id2tag[tid]

    prediction = {}
    for doc_id, pd in test_prepared.items():
        spans = decode_bio_to_char_spans(pd.word_spans, pred_tags_by_doc[doc_id])
        prediction[doc_id] = [[s, e, l] for (s, e, l) in spans]
    return prediction


def write_submission(prediction, out_dir, exp_name):
    pred_path = os.path.join(out_dir, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(prediction, f, ensure_ascii=False)

    zip_path = os.path.join(out_dir, f"submission_{exp_name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")  # must sit at the archive root
    print(f"Wrote {pred_path}")
    print(f"Wrote {zip_path}  (prediction.json is at the zip root, as required)")
    return zip_path


In [ ]:
# Example: package the best experiment found in section 11.
BEST_EXP_NAME = df.iloc[0]["experiment"]
print("Best experiment on dev:", BEST_EXP_NAME, "F1 =", df.iloc[0]["dev_strict_macro_f1"])

# Rebuild that experiment's architecture + load its best checkpoint, then predict on fold3.
# For "exp2_partial_ft", we use the base backbone with build_partial_ft_model.
backbone = load_phase1_backbone(PHASE1_ARCHS[0], vocab_size=len(tokenizer.itos),
                                 pad_token_id=PAD_ID, device=device)

best_model = build_partial_ft_model(backbone, NUM_LABELS, freeze_frac=0.5)
best_model.load_state_dict(torch.load(os.path.join(checkpoint_dir_p2, f"{BEST_EXP_NAME}_best.pt"),
                                       map_location="cpu"))
best_model.to(device)

prediction = predict_fold3(best_model, test_loader, test_prepared, device)
write_submission(prediction, phase2_dir, BEST_EXP_NAME)

Best experiment on dev: exp2_partial_ft F1 = 0.37055169382321074
[arch0_base] loaded /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/best_model_gqla_v1.pt | missing=0 unexpected=0
Wrote /content/drive/MyDrive/CS781/Capstone-Project/phase2_lner/prediction.json
Wrote /content/drive/MyDrive/CS781/Capstone-Project/phase2_lner/submission_exp2_partial_ft.zip  (prediction.json is at the zip root, as required)


'/content/drive/MyDrive/CS781/Capstone-Project/phase2_lner/submission_exp2_partial_ft.zip'

In [ ]:
backbone = load_phase1_backbone(PHASE1_ARCHS[0], vocab_size=len(tokenizer.itos),
                                 pad_token_id=PAD_ID, device=device)

EXP_NAME = "exp3_lora"

# Match model architecture to the experiment technique
expt_model = build_lora_model(backbone, NUM_LABELS, r=8, alpha=16)

expt_model.load_state_dict(torch.load(os.path.join(checkpoint_dir_p2, f"{EXP_NAME}_best.pt"),
                                       map_location="cpu"))
expt_model.to(device)

expt_prediction = predict_fold3(expt_model, test_loader, test_prepared, device)
write_submission(expt_prediction, phase2_dir, EXP_NAME)

[arch0_base] loaded /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/best_model_gqla_v1.pt | missing=0 unexpected=0
Wrote /content/drive/MyDrive/CS781/Capstone-Project/phase2_lner/prediction.json
Wrote /content/drive/MyDrive/CS781/Capstone-Project/phase2_lner/submission_exp3_lora.zip  (prediction.json is at the zip root, as required)


'/content/drive/MyDrive/CS781/Capstone-Project/phase2_lner/submission_exp3_lora.zip'

## 13. Submission checklist (Phase 2)

- [ ] `prediction.json` is directly at the root of `submission.zip` (no extra folder) - handled by `write_submission`.
- [ ] Every `fold3-text.json` document ID is present, no unknown IDs - `predict_fold3` iterates `test_prepared` which is built directly from `fold3-text.json`.
- [ ] Each entity prediction is `[start, end, label]` (char offsets, label from `ENTITY_LABELS`) - handled by `decode_bio_to_char_spans`.
- [ ] At least 6 separate Codabench submissions for Phase 2, covering at least 5 distinct fine-tuning/PEFT techniques + the mandatory KD run, **using all four Phase-1 architectures somewhere across the six** - run section 12 once per experiment in `results_table` and upload each zip.
- [ ] LORMA fine-tuning included (Experiment 4) - **re-check `LORMAInjectedLinear` against arXiv ref [12] before relying on it.**
- [ ] Report: discuss the "Expected Analysis" bullets from the brief's L-NER section (which technique/arch won, PEFT vs full-FT gap, entity types/boundaries that stay hard, effect of word-level BIO framing + chunking on long documents, class imbalance, KD quality-vs-efficiency trade-off, Phase-1 -> Phase-2 transfer).
